In [173]:
from collections import defaultdict

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.dagcircuit import DAGCircuit, DAGOpNode
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler.layout import Layout
from qiskit.circuit.quantumregister import Qubit

import matplotlib.pyplot as plt
import networkx as nx

from vfsexp import Vf

In [174]:
def extract_circuit(filename, path, node_num):
    # compose the input Quantum Circuit
    q = QuantumRegister(node_num, 'q')
    qc = QuantumCircuit(q)
    cir_temp = qc.from_qasm_file(path+filename)
    qc.compose(cir_temp, inplace=True)
    if node_num < cir_temp.num_qubits:
        raise Exception("Cannot compose circuit!") 

    return qc

""" From a QuantumCircuit, remove all 1qb gates and any 2qb gate [p,q] that has a parent [p,q] or [q,p]"""
def remove_1q_and_consecutive_2q_gates_in_circuit(circuit):

    newcirc = QuantumCircuit(circuit.num_qubits)
    record = [] #the current end layer of the reduced circuit
    for instruction in circuit:
        if len(instruction.qubits) != 2: continue
        if instruction.operation.name == 'cx': # only consider cx gates
            p, q = instruction.qubits[0]._index, instruction.qubits[1]._index
            if [p,q] in record or [q,p] in record:
                continue
            for x in record:
                if p in x or q in x:
                    record.remove(x)
            record.append([p,q])            
            newcirc.cx(p,q)
    return newcirc

def graph_of_circuit(circuit):
    ''' Return the induced graph of the reduced circuit C
            - node set: qubits in C
            - edge set: all pair (p,q) if CNOT [p,q] or CNOT[q,p] in C
        Args:
            C (list): the input reduced circuit
        Returns:
            g (Graph)
    '''   
    g = nx.Graph()
    for gate in circuit:
        if gate.operation.name != 'cx': continue
        p, q = gate.qubits[0]._index, gate.qubits[1]._index
        g.add_edge(p,q)
    return g  

#def is_embeddable(g, H, anchor, stop):
#    '''check if a small graph g is embeddable in a large H, anchor is bool
#        g, H (Graph)
#        anchor (bool): whether or not mapping anchor of g to that of H
#        stop (float): time limit for vf2
#    '''
#    vf2 = Vf()
#    result = {} 
#    if anchor: result[hub(g)] = hub(H)
#    result = vf2.dfsMatch(g, H, result, stop)
#    lng = len(nx.nodes(g))
#    if len(result) == lng:
#        return True, result   
#    return False, result

def has_cycle_3(graph):
    for u in graph.nodes():
        for v in graph.neighbors(u):
            for w in graph.neighbors(v):
                if u != w and graph.has_edge(u, w):
                    return True
    return False

def fast_no_embedding(subgraph, graph):
    # for AG with no 3-cycle
    if has_cycle_3(subgraph) and not has_cycle_3(graph): 
        return True
    
    subgraph_deg = list(nx.degree(subgraph, v) for v in subgraph.nodes())
    subgraph_deg.sort(reverse=True)
    
    graph_deg = list(nx.degree(subgraph, v) for v in subgraph.nodes())
    graph_deg.sort(reverse=True)
    
    l = len(subgraph_deg)
    # if l > len(graph_deg): 
    #     return True
    for idx in range(l):
        if subgraph_deg[idx] > graph_deg[idx]:
            return True
        
    return False

def is_embeddable(g, H, stop):
    '''check if a small graph g is embeddable in a large H, anchor is bool
        g, H (Graph)
        stop (float): time limit for vf2
    '''
    if fast_no_embedding(g, H):
        return False, {}
    
    lng = len(nx.nodes(g))
        
    new_g = g.copy()
    #print(len(new_g.edges()) == len(g.edges()))

    new_H = H.copy()
    temp_map = dict()

    vf2 = Vf(new_g, new_H, {}, stop)
    result = vf2.dfsMatch({})
    if len(result) == lng:
        return True, temp_map.update(result)  
    return False, {}

In [193]:
def _successors(node, dag):
    for _, successor, edge_data in dag.edges(node):
        if not isinstance(successor, DAGOpNode):
            continue
        if isinstance(edge_data, Qubit):
            yield successor

"""Partition a dagCircuit into sections so that each section is embeddable into a given architecture""" 
def partition(dag, architecture, method='greedy'):
    part_list = []
    
    canonical_register = dag.qregs["q"]

    front_layer = dag.front_layer() #sl This front layer contains 1qbg
    for node in front_layer:
        if len(node.qargs) != 2 or node.op.name != 'cx': 
            print(f'check why dag still has single-qubit gates {type(node), node.op.name}')
            return 
    
    # Create a defaultdict with a default value of 0
    applied_predecessors = defaultdict(int)
    for _, input_node in dag.input_map.items(): 
        for successor in _successors(input_node, dag):
            applied_predecessors[successor] += 1

    current_part_dag = dag.copy_empty_like()
    # generate the current layer and examine for each node of the current layer
    
    step = 0
    while front_layer:
        step += 1
        execute_gate_list = []
        
        progress = False # Check if we add new edges to current_part_dag
        for node in front_layer:
            
            v0, v1 = node.qargs

            # add node to current part if it 
            current_part_dag.apply_operation_back(node.op, node.qargs, node.cargs)
            temp_qc = dag_to_circuit(current_part_dag)
            temp_graph = graph_of_circuit(temp_qc)

            if is_embeddable(temp_graph, architecture, stop=10)[0]:
                execute_gate_list.append(node)
                #print(f'yes! {[edge for edge in temp_graph.edges()]}')
                progress = True
                continue
            
            # if not embeddable, put the node back 
            # print(f'Not embeddable! {[edge for edge in temp_graph.edges()]}')
            current_part_dag.remove_op_node(node)

        if progress and not execute_gate_list:
            raise Exception('Check why progress = True and not execute_gate_list!!')
            return
        
        if progress == False: # no new node in the current front layer can be added, 
                              # thus we end this section and start a new one
            if execute_gate_list: 
                raise Exception(f'!!check why progress {progress} and execute_gate_list is nonempty')
                return 
            
            print('constructed a new section!')
            
            part_list.append(current_part_dag)
            print([node.qargs for node in current_part_dag])

            # print the corresponding graph edges
            temp_qc = dag_to_circuit(current_part_dag)
            temp_graph = graph_of_circuit(temp_qc)
            print([edge for edge in temp_graph.edges()])
            
            #reset
            current_part_dag = dag.copy_empty_like()


        if execute_gate_list: 
            #print(f'!check progress {progress} and {execute_gate_list}')
            for node in execute_gate_list:
                front_layer.remove(node)
                for successor in _successors(node, dag):
                    applied_predecessors[successor] += 1
                    if applied_predecessors[node] == len(node.qargs):
                        front_layer.append(successor)
                        
            if not front_layer:
                print('all nodes have been examined and we finish here')
                if current_part_dag.nodes():
                    part_list.append(current_part_dag)
                    print(f'end type 1')
                    return part_list
                print(f'end type 2')
                return part_list

        if progress:
            print(f'!!!check if progress is true: progress = {progress} {not execute_gate_list}')
            part_list.append(current_part_dag)
            current_part_dag = dag.copy_empty_like()
            

    print(f'end type 3')
    return part_list

In [194]:
from ag import qgrid
import time
import os

path = '../bench/qiskit_circuit_benchmark/' 
filename = 'AND_6.qasm'

AG = qgrid(2,3)
architecture = qgrid(2,3)
#print(architecture.edges())
#g = nx.Graph()
#g.add_edges_from([(4, 5), (4, 3), (5, 3)])
#print(is_embeddable(g,AG,100))

qc = extract_circuit(filename, path, len(AG.nodes()))
qc.draw('mpl')

newcirc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)
#for x in newcirc:
#    print(x.qubits[0]._index,x.qubits[1]._index)
newcirc.draw('mpl')

dag = circuit_to_dag(newcirc)
#for _, input_node in dag.input_map.items(): 
#    print(type(input_node), input_node.wire)


X = partition(dag, architecture, method='greedy')

!!!check if progress is true: progress = True False
!!!check if progress is true: progress = True False
!!!check if progress is true: progress = True False
all nodes have been examined and we finish here
end type 1
